In [ ]:
# Install LangGraph — the core library for building graph-based AI workflows.
!pip install -q langgraph

In [ ]:
from IPython.display import Image              # Displays images (graph diagrams) inline in Jupyter
from enum import Enum                          # Used to define fixed named values (ticket categories, priorities)
from langchain_core.runnables import Runnable  # Base type for compiled LangGraph graphs
from langgraph.graph import END, START, StateGraph  # Core LangGraph building blocks
from pathlib import Path                       # Cross-platform file path handling
from typing import TypedDict                   # Defines the shape of the state dictionary


# Helper: renders the compiled graph as a PNG image and shows it inline in Jupyter
def display_graph(runnable: Runnable, output_png: Path):
    with output_png.open(mode="wb") as file:
        # draw_mermaid_png() generates a visual diagram from the graph structure
        file.write(runnable.get_graph().draw_mermaid_png())

    display(Image(output_png, format="png"))

In [ ]:
# Enums provide a fixed set of named values — safer than using raw strings,
# because typos cause errors at definition time rather than silently producing wrong results.
class Category(Enum):
    BILLING = "billing"    # Tickets about payments, invoices, refunds
    TECHNICAL = "tech"     # Tickets about bugs, crashes, errors
    SPAM = "spam"          # Unwanted or irrelevant messages

class Priority(Enum):
    LOW = 1     # Normal, non-urgent tickets
    NORMAL = 2  # Tickets that need attention but aren't critical
    HIGH = 3    # Urgent tickets requiring immediate human attention

# The graph state — this dictionary is passed between all nodes.
# TypedDict enforces the field names and their types at type-check time.
class Ticket(TypedDict):
    text: str            # The raw text of the incoming support ticket
    category: Category   # Assigned by the 'classify' node
    priority: Priority   # Assigned by the 'classify' node
    response: str        # The final reply, set by billing/technical/reject, possibly amended by human_review

In [ ]:
# --- Node functions ---
# Each function receives the full state (Ticket) and returns a PARTIAL dict
# with only the fields it needs to update. LangGraph merges the result back into the state.

def classify(state: Ticket) -> dict:
    # Lowercase the text for case-insensitive keyword matching
    text = state["text"].lower()

    # Assign a category based on keywords found in the ticket text
    if any(k in text for k in ["refund", "invoice", "charge", "payment"]):
        category = Category.BILLING
    elif any(k in text for k in ["crash", "bug", "error", "not working"]):
        category = Category.TECHNICAL
    else:
        # Default: no billing or technical keywords found — treat as spam
        category = Category.SPAM

    # Escalate to HIGH priority if the ticket contains urgency signals
    priority: Priority = Priority.HIGH if "urgent" in text or "!!!" in text else Priority.LOW
    print(f"[classify] -> {category} / {priority}")
    return {"category": category, "priority": priority}


def billing(state: Ticket) -> dict:
    # Handle billing-related tickets with a standard acknowledgement response
    return {"response": "Billing: we will review your invoice within 24h."}


def technical(state: Ticket) -> dict:
    # Handle technical/bug tickets by asking for additional info
    return {"response": "Technical: please share the error log; we are on it."}


def reject(state: Ticket) -> dict:
    # Auto-reject spam tickets — no human involvement needed
    return {"response": "(auto-rejected as spam)"}


def human_review(state: Ticket) -> dict:
    # Escalate high-priority tickets by appending a note to the existing response
    escalated = state["response"] + " [ESCALATED TO HUMAN AGENT]"
    return {"response": escalated}

In [ ]:
# Routing (conditional edge) function — decides what node to go to AFTER billing/technical run.
# Returns either "human_review" (escalate) or END (finish the workflow).
# The return value must match a registered node name or the special END sentinel.
def maybe_escalate(state: Ticket) -> str:
    # Escalate if the priority is HIGH or NORMAL, but never escalate spam
    if (state["priority"] == Priority.HIGH or state["priority"] == Priority.NORMAL) and not state["category"] == Category.SPAM:
        return "human_review"
    return END  # No escalation needed — graph execution ends here

In [ ]:
# Build the conditional ticket-routing graph
graph_builder = StateGraph(Ticket)

# Register all processing nodes
graph_builder.add_node("classify", classify)
graph_builder.add_node("billing", billing)
graph_builder.add_node("technical", technical)
graph_builder.add_node("reject", reject)
graph_builder.add_node("human_review", human_review)

# Map each Category enum to the matching processing node name.
# We use .value (e.g., "billing") because conditional edges use string keys for routing.
processing_nodes_map = { Category.BILLING: "billing", Category.TECHNICAL: "technical", Category.SPAM: "reject" }
processing_nodes_map = { key.value: value for key, value in processing_nodes_map.items() }

# Every ticket starts at 'classify'
graph_builder.add_edge(START, "classify")

# After 'classify', route to the matching processing node based on the category value
graph_builder.add_conditional_edges("classify", lambda x: x["category"].value, processing_nodes_map)

# After each processing node, apply the escalation logic:
# high/normal priority -> human_review, otherwise -> END
for processing_node in processing_nodes_map.values():
    graph_builder.add_conditional_edges(processing_node, maybe_escalate, ["human_review", END])

# After human review, the workflow always ends
graph_builder.add_edge("human_review", END)

# Compile the graph into a runnable object
graph = graph_builder.compile()

In [ ]:
# Render and display the graph as a visual diagram.
# You'll see the branching paths: classify -> billing/technical/reject -> (maybe) human_review -> END
display_graph(graph, Path("/content/graph.png"))

In [ ]:
# Test the graph with three different ticket scenarios:
# 1. Billing issue (low priority)  -> billing node -> END
# 2. Urgent technical crash (HIGH) -> technical node -> human_review -> END
# 3. Spam                          -> reject node -> END
examples = [
    "My last invoice has a wrong charge, please refund.",
    "URGENT!!! the app crashes on startup, bug?",
    "Buy cheap watches now"
]

for example in examples:
    print(f">> {example}")

    # Invoke the graph with only the raw ticket text;
    # the 'classify' node will populate category and priority
    result = graph.invoke({ "text": example })
    print(result["response"])
    print()